In [ ]:
"""
Software Reliability Prediction
Dataset: blue_mountain_supercomputer_monthly_failures_proper.csv
Full Analysis with Graphs, Heatmaps & Forecasting
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.tree import DecisionTreeRegressor
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")


# =====================================================
# 1️⃣ LOAD DATASET
# =====================================================

def load_dataset(path):

    df = pd.read_csv(path)
    print("Columns in dataset:", df.columns.tolist())

    if "Month" in df.columns:
        df = df.sort_values("Month")

    if "MonthlyFailures" in df.columns:
        df["CumulativeFailures"] = df["MonthlyFailures"].cumsum()
        X = df["CumulativeFailures"].values.reshape(-1, 1)
        y = df["MonthlyFailures"].values
    else:
        numeric_col = df.select_dtypes(include=np.number).columns[0]
        X = np.arange(len(df)).reshape(-1, 1)
        y = df[numeric_col].values

    return X, y


# =====================================================
# 2️⃣ EVALUATION FUNCTIONS
# =====================================================

def evaluate_model(model, X_test, y_test):
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    corr = pearsonr(y_test, pred)[0]
    return mae, rmse, corr


def cross_validate_model(train_func, X, y):
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    maes, rmses, corrs = [], [], []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = train_func(X_train, y_train)
        mae, rmse, corr = evaluate_model(model, X_test, y_test)

        maes.append(mae)
        rmses.append(rmse)
        corrs.append(corr)

    return np.mean(maes), np.mean(rmses), np.mean(corrs)


# =====================================================
# 3️⃣ MODELS
# =====================================================

def train_bpn(X, y):
    model = MLPRegressor(hidden_layer_sizes=(20,),
                         activation='tanh',
                         max_iter=2000)
    model.fit(X, y)
    return model


def train_rbfn(X, y):
    model = KernelRidge(kernel='rbf', alpha=1.0, gamma=0.1)
    model.fit(X, y)
    return model


def train_svm(X, y):
    model = SVR(kernel='rbf', C=100, epsilon=0.1)
    model.fit(X, y)
    return model


def train_ccnn(X, y):
    model = MLPRegressor(hidden_layer_sizes=(10, 10),
                         activation='tanh',
                         max_iter=2000)
    model.fit(X, y)
    return model


def train_decision_tree(X, y):
    model = DecisionTreeRegressor()
    model.fit(X, y)
    return model


# =====================================================
# 4️⃣ MAIN EXECUTION
# =====================================================

if __name__ == "__main__":

    path = "/kaggle/input/datasets/tungalavenkata/blue-main/blue_mountain_supercomputer_monthly_failures_proper.csv"

    X, y = load_dataset(path)

    models = {
        "Back Propagation NN": train_bpn,
        "RBF Network": train_rbfn,
        "Support Vector Machine": train_svm,
        "Cascade Correlation NN": train_ccnn,
        "Decision Tree": train_decision_tree
    }

    print("\n==============================")
    print(" SOFTWARE RELIABILITY RESULTS ")
    print("==============================\n")

    results = {}

    for name, func in models.items():
        mae, rmse, corr = cross_validate_model(func, X, y)
        results[name] = {"MAE": mae, "RMSE": rmse, "Correlation": corr}

    results_df = pd.DataFrame(results).T.sort_values("RMSE")

    print(results_df)

    best_model_name = results_df.index[0]
    print(f"\nBest Model (Lowest RMSE): {best_model_name}")

    # =====================================================
    # 📊 RMSE BAR PLOT
    # =====================================================

    plt.figure(figsize=(10,6))
    plt.bar(results_df.index, results_df["RMSE"])
    plt.xticks(rotation=45)
    plt.title("Model Comparison - RMSE")
    plt.ylabel("RMSE")
    plt.grid(axis='y')
    plt.show()

    # =====================================================
    # 📊 CORRELATION BAR PLOT
    # =====================================================

    plt.figure(figsize=(10,6))
    plt.bar(results_df.index, results_df["Correlation"])
    plt.xticks(rotation=45)
    plt.title("Model Comparison - Correlation")
    plt.ylabel("Correlation")
    plt.grid(axis='y')
    plt.show()

    # =====================================================
    # 📈 RELIABILITY GROWTH CURVE
    # =====================================================

    best_model = models[best_model_name](X, y)
    predictions = best_model.predict(X)

    plt.figure(figsize=(10,6))
    plt.plot(y, label="Actual Failures", marker='o')
    plt.plot(predictions, label=f"Predicted ({best_model_name})", linestyle='--')
    plt.title("Reliability Growth Curve")
    plt.xlabel("Month Index")
    plt.ylabel("Failures")
    plt.legend()
    plt.grid()
    plt.show()

    # =====================================================
    # 📉 RESIDUAL PLOT
    # =====================================================

    residuals = y - predictions

    plt.figure(figsize=(10,6))
    plt.scatter(predictions, residuals)
    plt.axhline(0, linestyle='--')
    plt.title("Residual Plot")
    plt.xlabel("Predicted Failures")
    plt.ylabel("Residual Error")
    plt.grid()
    plt.show()

    # =====================================================
    # 📆 FUTURE 6 MONTH FORECAST
    # =====================================================

    last_value = X[-1][0]
    future_X = np.arange(last_value+1, last_value+7).reshape(-1,1)
    future_pred = best_model.predict(future_X)

    print("\nFuture 6-Month Predictions:")
    for i, val in enumerate(future_pred, 1):
        print(f"Month +{i}: {val:.2f}")

    plt.figure(figsize=(10,6))
    plt.plot(range(len(y)), y, label="Historical")
    plt.plot(range(len(y), len(y)+6), future_pred,
             marker='o', color='red', label="Next 6 Months")
    plt.title("Future Failure Forecast")
    plt.xlabel("Month Index")
    plt.ylabel("Predicted Failures")
    plt.legend()
    plt.grid()
    plt.show()

    # =====================================================
    # 📊 CORRELATION MATRIX + HEATMAP
    # =====================================================

    prediction_dict = {"Actual": y}

    for name, func in models.items():
        prediction_dict[name] = func(X, y).predict(X)

    corr_df = pd.DataFrame(prediction_dict)
    correlation_matrix = corr_df.corr()

    print("\nCorrelation Matrix:\n")
    print(correlation_matrix)

    plt.figure(figsize=(10,8))
    sns.heatmap(correlation_matrix,
                annot=True,
                cmap="coolwarm",
                fmt=".2f",
                linewidths=0.5)
    plt.title("Model Prediction Correlation Matrix")
    plt.show()


Columns in dataset: ['SMP', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15']

 SOFTWARE RELIABILITY RESULTS 



In [ ]:
"""
Software Reliability Prediction
Dataset: blue_mountain_simulated_failure_times_corrected.csv
(Kaggle Path Version)
Format: SMP + timestamps of failures
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.tree import DecisionTreeRegressor
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")


# =====================================================
# 1️⃣ LOAD & CONVERT TIMESTAMP DATASET
# =====================================================

def load_timestamp_dataset(path):

    df = pd.read_csv(path)
    print("Dataset Loaded Successfully")
    print("Columns:", df.columns.tolist())

    # Extract all timestamps (ignore SMP column)
    all_timestamps = df.iloc[:, 1:].values.flatten()
    all_timestamps = all_timestamps[~pd.isna(all_timestamps)]

    max_time = int(np.ceil(max(all_timestamps)))
    print("Detected Total Months:", max_time)

    monthly_failures = np.zeros(max_time)

    for _, row in df.iterrows():
        timestamps = row[1:].dropna().values
        for t in timestamps:
            month_index = int(np.ceil(t)) - 1
            if 0 <= month_index < max_time:
                monthly_failures[month_index] += 1

    cumulative_failures = np.cumsum(monthly_failures)

    X = cumulative_failures.reshape(-1, 1)
    y = monthly_failures

    return X, y


# =====================================================
# 2️⃣ EVALUATION
# =====================================================

def evaluate_model(model, X_test, y_test):
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    corr = pearsonr(y_test, pred)[0]
    return mae, rmse, corr


def cross_validate_model(train_func, X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    maes, rmses, corrs = [], [], []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = train_func(X_train, y_train)
        mae, rmse, corr = evaluate_model(model, X_test, y_test)

        maes.append(mae)
        rmses.append(rmse)
        corrs.append(corr)

    return np.mean(maes), np.mean(rmses), np.mean(corrs)


# =====================================================
# 3️⃣ MODELS
# =====================================================

def train_bpn(X, y):
    model = MLPRegressor(hidden_layer_sizes=(20,),
                         activation='tanh',
                         max_iter=2000,
                         random_state=42)
    model.fit(X, y)
    return model


def train_rbfn(X, y):
    model = KernelRidge(kernel='rbf', alpha=1.0, gamma=0.1)
    model.fit(X, y)
    return model


def train_svm(X, y):
    model = SVR(kernel='rbf', C=100, epsilon=0.1)
    model.fit(X, y)
    return model


def train_ccnn(X, y):
    model = MLPRegressor(hidden_layer_sizes=(10, 10),
                         activation='tanh',
                         max_iter=2000,
                         random_state=42)
    model.fit(X, y)
    return model


def train_decision_tree(X, y):
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X, y)
    return model


# =====================================================
# 4️⃣ MAIN EXECUTION
# =====================================================

if __name__ == "__main__":

    path = "/kaggle/input/datasets/tungalavenkata/blue-mountain-csv/blue_mountain_simulated_failure_times_corrected.csv"

    X, y = load_timestamp_dataset(path)

    models = {
        "Back Propagation NN": train_bpn,
        "RBF Network": train_rbfn,
        "Support Vector Machine": train_svm,
        "Cascade Correlation NN": train_ccnn,
        "Decision Tree": train_decision_tree
    }

    print("\n==============================")
    print(" SOFTWARE RELIABILITY RESULTS ")
    print("==============================\n")

    results = {}

    for name, func in models.items():
        mae, rmse, corr = cross_validate_model(func, X, y)
        results[name] = {"MAE": mae, "RMSE": rmse, "Correlation": corr}

    results_df = pd.DataFrame(results).T.sort_values("RMSE")
    print(results_df)

    best_model_name = results_df.index[0]
    print(f"\nBest Model (Lowest RMSE): {best_model_name}")

    # =====================================================
    # 📊 RMSE BAR PLOT
    # =====================================================

    plt.figure(figsize=(10,6))
    plt.bar(results_df.index, results_df["RMSE"])
    plt.xticks(rotation=45)
    plt.title("Model Comparison - RMSE")
    plt.ylabel("RMSE")
    plt.grid(axis='y')
    plt.show()

    # =====================================================
    # 📈 RELIABILITY GROWTH CURVE
    # =====================================================

    best_model = models[best_model_name](X, y)
    predictions = best_model.predict(X)

    plt.figure(figsize=(10,6))
    plt.plot(y, marker='o', label="Actual Failures")
    plt.plot(predictions, linestyle='--', label="Predicted")
    plt.title("Reliability Growth Curve")
    plt.xlabel("Month")
    plt.ylabel("Failures")
    plt.legend()
    plt.grid()
    plt.show()

    # =====================================================
    # 📆 FUTURE FORECAST
    # =====================================================

    last_val = X[-1][0]
    future_X = np.arange(last_val+1, last_val+7).reshape(-1,1)
    future_pred = best_model.predict(future_X)

    print("\nFuture 6-Month Predictions:")
    for i, val in enumerate(future_pred, 1):
        print(f"Month +{i}: {val:.2f}")

    plt.figure(figsize=(10,6))
    plt.plot(range(len(y)), y, label="Historical")
    plt.plot(range(len(y), len(y)+6), future_pred,
             marker='o', color='red', label="Next 6 Months")
    plt.title("Future Failure Forecast")
    plt.legend()
    plt.grid()
    plt.show()

    # =====================================================
    # 📊 CORRELATION MATRIX + HEATMAP
    # =====================================================

    prediction_dict = {"Actual": y}

    for name, func in models.items():
        prediction_dict[name] = func(X, y).predict(X)

    corr_df = pd.DataFrame(prediction_dict)
    correlation_matrix = corr_df.corr()

    print("\nCorrelation Matrix:\n")
    print(correlation_matrix)

    plt.figure(figsize=(10,8))
    sns.heatmap(correlation_matrix,
                annot=True,
                cmap="coolwarm",
                fmt=".2f",
                linewidths=0.5)
    plt.title("Model Prediction Correlation Matrix")
    plt.show()


In [ ]:
"""
Unified Software Reliability Regression Benchmark
Combines:
 - Paper 1 models (PNN, GRNN, KNN, DT, SVR, Bagging, Linear)
 - Paper 2 models (BPNN, CCNN, RBFN)
Dataset: blue_mountain_supercomputer_monthly_failures_proper.csv
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.kernel_ridge import KernelRidge


# =========================================================
# Kernel regressors (PNN / GRNN)
# =========================================================
class PNNRegressor:
    def __init__(self, sigma=0.5):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x-xi)**2)/(2*self.sigma**2))

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            w = np.array([self._gauss(x, xi) for xi in self.X])
            if w.sum() == 0:
                preds.append(np.mean(self.y))
            else:
                preds.append(np.sum(w*self.y)/np.sum(w))
        return np.array(preds)


class GRNN:
    def __init__(self, sigma=0.5):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x-xi)**2)/(2*self.sigma**2))

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            w = np.array([self._gauss(x, xi) for xi in self.X])
            if w.sum() == 0:
                preds.append(np.mean(self.y))
            else:
                preds.append(np.sum(w*self.y)/np.sum(w))
        return np.array(preds)


# =========================================================
# Load SMP monthly dataset
# =========================================================
file_path = "/kaggle/input/datasets/tungalavenkata/blue-main/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

# Drop device id if present
if "SMP" in df.columns:
    df = df.drop(columns=["SMP"])

# Predict last month using previous months
target_col = df.columns[-1]
feature_cols = df.columns[:-1]

X = df[feature_cols].values.astype(float)
y = df[target_col].values.astype(float)

print("Predicting:", target_col)
print("Features:", feature_cols.tolist())
print("Samples:", len(X))


# =========================================================
# Regression models (ALL)
# =========================================================
models = {
    # Paper 1
    "PNN": PNNRegressor(0.5),
    "GRNN": GRNN(0.5),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree": DecisionTreeRegressor(),
    "SVR": SVR(kernel="rbf", C=100, epsilon=0.1),
    "Bagging": BaggingRegressor(n_estimators=50, random_state=42),
    "Linear Regression": LinearRegression(),

    # Paper 2
    "Backprop NN": MLPRegressor(hidden_layer_sizes=(20,),
                                activation="tanh",
                                max_iter=2000),

    "Cascade NN": MLPRegressor(hidden_layer_sizes=(10,10),
                               activation="tanh",
                               max_iter=2000),

    "RBF Network": KernelRidge(kernel="rbf", alpha=1.0, gamma=0.1),
}


# =========================================================
# Cross-validation
# =========================================================
def evaluate(model, X, y):
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    rmses, maes, r2s = [], [], []

    for tr, te in kf.split(X):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model.fit(Xtr, ytr)
        pred = model.predict(Xte)

        rmses.append(np.sqrt(mean_squared_error(yte, pred)))
        maes.append(mean_absolute_error(yte, pred))
        r2s.append(r2_score(yte, pred))

    return np.mean(rmses), np.mean(maes), np.mean(r2s)


# =========================================================
# Run all models
# =========================================================
results = []

for name, model in models.items():
    try:
        rmse, mae, r2 = evaluate(model, X, y)
        results.append([name, rmse, mae, r2])
    except Exception as e:
        results.append([name, None, None, str(e)])

results_df = pd.DataFrame(results,
                          columns=["Model", "RMSE", "MAE", "R2"])\
                          .sort_values("RMSE")

print("\n===== Regression Benchmark Results =====\n")
print(results_df.to_string(index=False))

In [ ]:
# benchmark_with_smp_handling.py
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.kernel_ridge import KernelRidge

# ---------- simple PNN / GRNN ----------
class PNNRegressor:
    def __init__(self, sigma=0.5):
        self.sigma = sigma
    def _gauss(self, x, xi):
        return np.exp(-np.sum((x-xi)**2)/(2*self.sigma**2))
    def fit(self, X, y):
        self.X = np.asarray(X)
        self.y = np.asarray(y)
    def predict(self, X):
        X = np.atleast_2d(X)
        preds = []
        for x in X:
            w = np.array([self._gauss(x, xi) for xi in self.X])
            preds.append(np.sum(w*self.y)/w.sum() if w.sum()!=0 else np.mean(self.y))
        return np.array(preds)

class GRNN(PNNRegressor):
    pass

# ---------- Load dataset ----------
file_path = "/kaggle/input/datasets/tungalavenkata/blue-main/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

# Keep SMP id for analysis, drop it for features
if "SMP" in df.columns:
    ids = df["SMP"].astype(str).values
    df_features = df.drop(columns=["SMP"])
else:
    ids = np.arange(len(df)).astype(str)
    df_features = df.copy()

target_col = df_features.columns[-1]        # last month
feature_cols = df_features.columns[:-1]     # months 1..(T-1)

X = df_features[feature_cols].values.astype(float)
y = df_features[target_col].values.astype(float)

print("Samples (SMP machines):", len(X))
print("Months used as features:", list(feature_cols))
print("Target month:", target_col)

# ---------- Models ----------
models = {
    "PNN": PNNRegressor(0.5),
    "GRNN": GRNN(0.5),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "SVR": SVR(kernel="rbf", C=100, epsilon=0.1),
    "Bagging": BaggingRegressor(n_estimators=50, random_state=42),
    "Linear Regression": LinearRegression(),
    "Backprop NN": MLPRegressor(hidden_layer_sizes=(20,), activation="tanh", max_iter=2000, random_state=42),
    "Cascade NN": MLPRegressor(hidden_layer_sizes=(10,10), activation="tanh", max_iter=2000, random_state=42),
    "RBF Network": KernelRidge(kernel="rbf", alpha=1.0, gamma=0.1),
}

# ---------- evaluation helpers ----------
def evaluate_kfold(model, X, y, n_splits=10):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmses, maes, r2s = [], [], []
    for tr, te in kf.split(X):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)
        model.fit(Xtr_s, ytr)
        pred = model.predict(Xte_s)
        rmses.append(np.sqrt(mean_squared_error(yte, pred)))
        maes.append(mean_absolute_error(yte, pred))
        r2s.append(r2_score(yte, pred))
    return np.mean(rmses), np.mean(maes), np.mean(r2s)

def cross_val_predictions(model, X, y, n_splits=10):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    preds = np.empty_like(y, dtype=float)
    for tr, te in kf.split(X):
        Xtr, Xte = X[tr], X[te]
        ytr = y[tr]
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)
        model.fit(Xtr_s, ytr)
        preds[te] = model.predict(Xte_s)
    return preds

def evaluate_loso(model, X, y):
    loo = LeaveOneOut()
    rmses, maes, r2s = [], [], []
    for tr, te in loo.split(X):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)
        model.fit(Xtr_s, ytr)
        pred = model.predict(Xte_s)
        rmses.append(np.sqrt(mean_squared_error(yte, pred)))
        maes.append(mean_absolute_error(yte, pred))
        r2s.append(r2_score(yte, pred))
    return np.mean(rmses), np.mean(maes), np.mean(r2s)

# ---------- Run benchmarks ----------
results = []
cv_preds_per_model = {}

for name, model in models.items():
    try:
        rmse, mae, r2 = evaluate_kfold(model, X, y, n_splits=10)
        results.append([name, rmse, mae, r2])
        # also store CV predictions for later per-SMP error analysis (best-effort)
        try:
            preds = cross_val_predictions(model, X, y, n_splits=10)
            cv_preds_per_model[name] = preds
        except Exception:
            cv_preds_per_model[name] = None
    except Exception as e:
        results.append([name, None, None, str(e)])
        cv_preds_per_model[name] = None

results_df = pd.DataFrame(results, columns=["Model", "RMSE", "MAE", "R2"]).sort_values("RMSE")
print("\nResults (10-fold KFold across SMP rows):")
print(results_df.to_string(index=False))

# ---------- per-SMP errors for best model (if CV preds available) ----------
best = results_df[results_df['RMSE'].notna()].iloc[0]['Model']
print("\nBest model by RMSE:", best)
preds = cv_preds_per_model.get(best)
if preds is not None:
    per_smp_abs_err = np.abs(y - preds)
    per_smp_df = pd.DataFrame({
        "SMP": ids,
        "Actual": y,
        "Pred": preds,
        "AbsErr": per_smp_abs_err
    }).sort_values("AbsErr", ascending=False)
    print("\nTop 8 SMPs by absolute error (worst first):")
    print(per_smp_df.head(8).to_string(index=False))
    print("\nMean per-SMP absolute error:", per_smp_df["AbsErr"].mean())
else:
    print("No CV predictions available for best model (couldn't compute predictions).")
